# UCS420 Assignment 8 — NLP using Python-II
**Name:** Aastha Tyagi  
**Roll No.:** 1024160131  
Feature extraction, sentiment analysis and text generation.


In [ ]:
import nltk, re, string
import pandas as pd
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from IPython.display import display

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "vader_lexicon"]:
    nltk.download(pkg, quiet=True)
print("ready")


## Q1. Paragraph, cleaning, tokenization, stopwords, frequency

Favourite topic: **hostel badminton at Thapar**.


In [ ]:
paragraph = (
    "On most evenings I walk from the academic block to the indoor badminton courts. "
    "The wooden floor smells faintly of resin, and the only soundtrack is the sharp ping of a smash. "
    "I started playing in school because it was the one sport that did not need a large team. "
    "Now I care more about footwork than about winning every rally. "
    "A slow serve can still beat a flashy jump smash if the opponent is off balance. "
    "That mix of patience and sudden speed is why badminton stays my favourite topic."
)
print(paragraph)
print("\nSentence count:", len(sent_tokenize(paragraph)))

lowered = paragraph.lower()
no_punct = re.sub(r"[^a-z0-9\s]", " ", lowered)
print("\nLowercase + punctuation removed:\n", no_punct)

split_tokens = no_punct.split()
nltk_tokens = word_tokenize(no_punct)
print("\nsplit()        :", split_tokens)
print("word_tokenize():", nltk_tokens)
print("split vs NLTK extra/missing:", set(nltk_tokens) - set(split_tokens), set(split_tokens) - set(nltk_tokens))
print(
    "Difference: str.split() only cuts on whitespace. word_tokenize() also splits clitics and "
    "punctuation into separate tokens when punctuation is still present. After we stripped "
    "punctuation with regex, both lists are close; NLTK remains the safer default for raw text."
)

stop = set(stopwords.words("english"))
content = [w for w in nltk_tokens if w not in stop]
print("\nWithout stopwords:", content)
freq = Counter(content)
print("Frequency:", dict(freq.most_common()))


## Q2. Regex alphabetic words, stemming vs lemmatization


In [ ]:
alpha_words = re.findall(r"\b[a-zA-Z]+\b", paragraph)
print("Alphabetic words:", alpha_words)
filtered = [w.lower() for w in alpha_words if w.lower() not in stop]
ps, wnl = PorterStemmer(), WordNetLemmatizer()
stemmed = [ps.stem(w) for w in filtered]
lemmatized = [wnl.lemmatize(w) for w in filtered]
display(pd.DataFrame(list(zip(filtered, stemmed, lemmatized)),
                     columns=["word", "Porter stem", "WordNet lemma"]))
print(
    "When to prefer stemming: search indexes and noisy lab tickets, where collapsing "
    "playing/played/plays to one key is more useful than linguistic correctness. "
    "When to prefer lemmatization: any task that must keep real English words "
    "(chatbots, knowledge bases). Example in this paragraph: 'rallies' would become "
    "'ralli' with Porter but 'rally' with a POS-aware lemmatizer."
)


## Q3. Bag of Words and TF-IDF on three short texts


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

texts = [
    "Thapar library stays open late during mid-semester tests and students fill every desk.",
    "Campus Wi-Fi drops in the lecture halls whenever too many laptops join the same access point.",
    "The hostel mess served extra paneer on Friday and the queue stretched past the notice board.",
]
print("Documents:")
for i, t in enumerate(texts, 1):
    print(f" D{i}: {t}")

cv = CountVectorizer()
bow = cv.fit_transform(texts)
bow_df = pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out(), index=["D1", "D2", "D3"])
print("\nBag of Words:")
display(bow_df)

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(texts)
names = tfidf.get_feature_names_out()
tfidf_df = pd.DataFrame(X.toarray(), columns=names, index=["D1", "D2", "D3"])
print("TF-IDF:")
display(tfidf_df.round(3))

print("\nTop 3 keywords per document (TF-IDF):")
for i, row in enumerate(X.toarray()):
    idx = np.argsort(row)[::-1][:3]
    print(f" D{i+1}:", [(names[j], round(row[j], 3)) for j in idx])
print(
    "Interpretation: TF-IDF down-weights words that appear in every campus note and "
    "promotes distinctive terms — library/desk, Wi-Fi/laptops, paneer/mess."
)


## Q4. Jaccard vs cosine similarity (AI vs Blockchain)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

ai_text = (
    "Artificial intelligence learns patterns from data so software can classify images, "
    "rank search results and draft text. Modern systems mix statistical models with large "
    "neural networks trained on many examples. The useful output is a prediction or a generated sentence. "
    "Engineers still have to check bias, energy cost and whether the model fails on rare cases. "
    "AI tools sit inside cameras, chat apps and hospital triage screens. "
    "They improve when more labelled data arrives, not when a ledger grows."
)
bc_text = (
    "Blockchain is a shared ledger where each block stores hashed records that later blocks confirm. "
    "No single office owns the history, so changing an old payment is hard without majority agreement. "
    "Cryptocurrencies, supply-chain stamps and certificate registries use this idea. "
    "The network spends energy on consensus rather than on training a neural net. "
    "Smart contracts run small programs when a condition on the chain is met. "
    "The core promise is tamper-evident history, not language generation."
)
print("AI text:\n", ai_text)
print("\nBlockchain text:\n", bc_text)

def content_tokens(text):
    toks = [w.lower() for w in word_tokenize(re.sub(r"[^a-zA-Z\s]", " ", text))]
    return [t for t in toks if t not in stop and t.isalpha()]

s1, s2 = set(content_tokens(ai_text)), set(content_tokens(bc_text))
jaccard = len(s1 & s2) / len(s1 | s2)
print("\nJaccard:", round(jaccard, 4), " overlap:", sorted(s1 & s2)[:20])

vec = TfidfVectorizer(stop_words="english")
mat = vec.fit_transform([ai_text, bc_text])
cos = cosine_similarity(mat[0], mat[1])[0, 0]
print("Cosine :", round(float(cos), 4))
print(
    "In this pair Jaccard is low because the two technologies share few content words "
    "(data, network). Cosine on TF-IDF is also low, but it still notices overlapping "
    "function-ish stems less noisily after IDF. Cosine is the better insight here: it "
    "uses weights, so a repeated generic word does not dominate, while Jaccard treats "
    "every shared type as equal and ignores frequency."
)


## Q5. Sentiment (VADER) and word cloud of positive reviews


In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

reviews = [
    "The noise-cancelling earbuds survived a month of bus rides and the bass is clean.",
    "Charging case started rattling on day three and one bud disconnects in the rain.",
    "Delivery was on time. Sound is okay for lectures, nothing special.",
    "Battery easily lasts a full lab day and the fit stays comfortable.",
    "Customer care replaced the left bud in two days. Really grateful.",
]
sia = SentimentIntensityAnalyzer()
rows = []
for r in reviews:
    s = sia.polarity_scores(r)
    if s["compound"] >= 0.05:
        label = "Positive"
    elif s["compound"] <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    rows.append({"review": r, **s, "class": label})
sent_df = pd.DataFrame(rows)
display(sent_df)

positive_text = " ".join(sent_df.loc[sent_df["class"] == "Positive", "review"])
wc = WordCloud(width=800, height=400, background_color="white").generate(positive_text)
plt.figure(figsize=(8, 4))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word cloud — positive reviews")
plt.show()


## Q6. Tiny Dense next-word model (Keras Tokenizer)


In [ ]:
train_paragraph = (
    "Badminton rewards players who watch the shuttle early and move their feet before swinging. "
    "A high serve pulls the opponent back; a drop shot then invites a weak return. "
    "Practice sessions at the indoor court start with warm-up rallies and end with match points. "
    "Coaches tell beginners to keep the grip relaxed so the wrist can snap. "
    "Doubles needs fast communication, while singles is a longer chess game on a smaller court."
)
print(train_paragraph)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten
import numpy as np

tokenizer = Tokenizer()
tokenizer.fit_on_texts([train_paragraph.lower()])
total_words = len(tokenizer.word_index) + 1
print("Vocabulary:", total_words)

input_sequences = []
for line in train_paragraph.lower().split("."):
    line = line.strip()
    if not line:
        continue
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        input_sequences.append(tokens[: i + 1])
max_len = max(len(s) for s in input_sequences)
sequences = pad_sequences(input_sequences, maxlen=max_len, padding="pre")
X, y = sequences[:, :-1], to_categorical(sequences[:, -1], num_classes=total_words)
print("Training pairs:", len(X), "max_len:", max_len)

model = Sequential([
    Embedding(total_words, 24, input_length=max_len - 1),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(total_words, activation="softmax"),
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X, y, epochs=200, verbose=0)
print("training finished")

def generate_line(seed, n_words=8):
    text = seed.lower()
    for _ in range(n_words):
        seq = tokenizer.texts_to_sequences([text])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding="pre")
        nxt = int(np.argmax(model.predict(seq, verbose=0), axis=-1)[0])
        word = next((w for w, i in tokenizer.word_index.items() if i == nxt), None)
        if not word:
            break
        text += " " + word
    return text

print("Generated 1:", generate_line("badminton", 10))
print("Generated 2:", generate_line("practice", 10))
print("Generated 3:", generate_line("doubles", 8))
